# PatchTST trên PTB-XL: Membership Inference (TV2, Hoàng)

Chạy trọn pipeline của `experiment/patchtst/patchtst_mia.py` cho cả **cls** và **fcst**:
tune (≤ 6 cấu hình, chỉ nhìn `val`) → target → 8 shadow → Loss/Conf/Entropy, LiRA, RMIA, Temporal → `results/patchtst/<task>/*.json`.

**Trước khi chạy**
1. Tài khoản Kaggle đã xác minh số điện thoại (Settings → Phone Verification). Chưa xác minh thì không chọn được GPU và không bật được Internet.
2. Panel phải → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
3. *Add Input* → dataset `khyeh0719/ptb-xl-dataset` (notebook tự tìm `ptbxl_database.csv` trong `/kaggle/input`).
4. Để chạy không cần mở máy: **Save Version → Save & Run All (Commit)**. Bản commit chạy nền tối đa 12 giờ, xong thì vào tab *Output* tải `patchtst_results.zip`.

Hai GPU: cls chạy trên GPU 0, fcst trên GPU 1 cùng lúc. Nếu chỉ có 1 GPU thì hai task chạy lần lượt.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import glob, os
print(glob.glob('/kaggle/input/**/ptbxl_database.csv', recursive=True))

## 1. Lấy repo nhóm (split + shadow mask dùng chung) và ghi script PatchTST

In [ ]:
REPO = '/kaggle/working/MIA'
if not os.path.exists(REPO):
    !git clone -q https://github.com/pearngoc/MIA.git {REPO}
os.makedirs(f'{REPO}/experiment/patchtst', exist_ok=True)
!ls {REPO}/dataset/splits

In [ ]:
%%writefile /kaggle/working/MIA/experiment/patchtst/patchtst_mia.py
"""PatchTST on PTB-XL: target training, shadow models and every MIA attack (TV2).

Follows dataset/README.md (shared config for all models) and mirrors the MOMENT notebooks
(experiment/moment_*.ipynb) so the numbers are comparable:

  * data   : PTB-XL 100 Hz, single-superclass records, shared split + shadow masks (sha256 checked)
  * input  : 12 leads, whole record interpolated 1000 -> 608 points, z-score per lead per record,
             first 512 points = input; fcst target = last 96 points; cls uses only the first 512
  * train  : batch 16, AdamW, OneCycleLR (lr0 = max_lr / 25), grad clip 1.0, last epoch, no early stop,
             PatchTST trained from scratch; only (epochs, max_lr) are tuned, on val utility, <= 6 configs
  * attacks: cls  -> loss, conf, entropy, lira, lira_offline, rmia, rmia_online, temporal
             fcst -> loss, lira, lira_offline, rmia, rmia_online, temporal
  * output : results/patchtst/<task>/<attack>.json, schema of results/_sample.json

Steps (each one resumes / skips what is already done):
    python patchtst_mia.py tune    --task cls     # <= 6 (epochs, max_lr) configs, val utility only
    python patchtst_mia.py target  --task cls     # target model + loss/conf/entropy JSON
    python patchtst_mia.py shadows --task cls     # K = 8 shadow models on the shared masks
    python patchtst_mia.py attacks --task cls     # lira, lira_offline, rmia, rmia_online, temporal JSON
    python patchtst_mia.py all     --task cls     # the four steps in order
    python patchtst_mia.py smoke   --task cls     # 1-epoch run on a few hundred records to test the setup

Score convention everywhere: higher = more likely member.
"""
import argparse
import glob
import hashlib
import json
import math
import os
import random
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import interpolate
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc as sk_auc
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- shared constants (dataset/README.md)
SEED = 42
LEVEL = 'record'
N_LEADS = 12
RAW_LEN = 1000
LOOKBACK = 512
HORIZON = 96
BATCH = 16
GRAD_CLIP = 1.0
N_BOOT = 1000
N_FOLDS = 5
N_BANDS = 8
EVAL_FRACS = (0.1, 0.2, 0.5, 1.0)
DEFAULT_GRID = [(20, 1e-4), (20, 5e-4), (20, 1e-3), (50, 1e-4), (50, 5e-4), (50, 1e-3)]   # 6 configs max

# PatchTST architecture (Nie et al. 2023, supervised setting); fixed in advance, never tuned on MIA
ARCH = dict(num_input_channels=N_LEADS, context_length=LOOKBACK, patch_length=16, patch_stride=8,
            d_model=128, num_attention_heads=16, num_hidden_layers=3, ffn_dim=256,
            dropout=0.2, head_dropout=0.2, scaling='std')
BACKBONE = 'PatchTST (HF transformers, trained from scratch)'

# Every difference from dataset/README.md sections 2 (preprocessing) and 3 (training), written into each JSON
DEVIATIONS = [
    'Mục 2: z-score chia cho (std + 1e-8) thay vì std, để tránh chia 0 ở lead phẳng; ảnh hưởng không đáng kể.',
    "Mục 2: PatchTST có RevIN bên trong model (scaling='std'): chuẩn hóa input từng mẫu và hoàn nguyên output fcst. "
    'Đây là một phần kiến trúc (Nie et al. 2023), áp dụng SAU tiền xử lý chung, không thay tiền xử lý chung.',
    'Mục 2: đọc PTB-XL bằng loader riêng (wfdb, dự phòng bằng trình đọc WFDB format 16 viết bằng numpy), không dùng momentfm PTBXL_dataset; '
    'cùng bộ lọc single-superclass của make_split.py và cùng nội suy tuyến tính 1000 -> 608. Thứ tự bản ghi theo ecg_id tăng dần.',
    'Mục 3: kiến trúc cố định trước, không tune: patch 16, stride 8, d_model 128, 16 head, 3 layer, FFN 256, dropout 0.2, head_dropout 0.2.',
    'Mục 3: AdamW dùng weight_decay mặc định của PyTorch (0.01), giống notebook MOMENT; không thêm regularization nào khác.',
    'Mục 3: precision fp32 vì GPU Kaggle T4 không hỗ trợ bf16 (đúng quy tắc "fp32 nếu GPU cũ"); A100/L4 sẽ tự dùng bf16.',
    'Mục 3: lưới tune 6 cấu hình epochs {20, 50} x max_lr {1e-4, 5e-4, 1e-3}; mỗi cấu hình train trên toàn bộ member với seed 42, '
    'đánh giá ở epoch cuối trên val; chọn cấu hình val tốt nhất (cls: accuracy cao nhất, fcst: MSE thấp nhất).',
]
SELECTION_NOTE = ('Không có bước chọn hyperparameter hay checkpoint nào nhìn vào điểm MIA hoặc non-member của test: '
                  'tune chỉ dùng utility trên train/val (xem extra.tuning), điểm MIA chỉ được tính sau khi target và shadow đã train xong, '
                  'shadow dùng đúng recipe cuối của target, tham số attack (tau, gamma, a, HistGradientBoosting) cố định trước.')


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def control_randomness(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def save_atomic(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


# ---------------------------------------------------------------- data
def find_ptbxl(root):
    # shallow search only: a recursive glob walks ~44k record files and takes minutes on /kaggle/input
    hits = sorted((h for d in range(5) for h in glob.glob(os.path.join(root, *['*'] * d, 'ptbxl_database.csv'))), key=len)
    if not hits:
        sys.exit(f'ptbxl_database.csv not found under {root}; pass --ptbxl-dir')
    return os.path.dirname(hits[0])


def _read_wfdb_numpy(path):
    """Minimal reader for PTB-XL WFDB records (format 16), used when the wfdb package is missing."""
    with open(path + '.hea') as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    n_sig, n_samp = int(lines[0].split()[1]), int(lines[0].split()[3])
    gains, bases, dat = [], [], None
    for l in lines[1:1 + n_sig]:
        p = l.split()
        dat = p[0]
        assert p[1] == '16', f'unsupported WFDB format {p[1]} in {path}'
        m = re.match(r'([-\d.eE+]+)(?:\(([-\d]+)\))?', p[2])
        gains.append(float(m.group(1)))
        bases.append(float(m.group(2)) if m.group(2) is not None else float(p[4]))
    raw = np.fromfile(os.path.join(os.path.dirname(path), dat), dtype='<i2').reshape(n_samp, n_sig)
    return ((raw - np.array(bases)) / np.array(gains)).T


def read_record(path):
    try:
        import wfdb
        sig, _ = wfdb.rdsamp(path, channels=list(range(N_LEADS)))
        return sig.T
    except ImportError:
        return _read_wfdb_numpy(path)


def load_pool(args):
    """All single-label records in the order of the shared mask file (sorted ecg_id).

    Returns dict with ids, y, x [N, 12, 608] float32, index arrays of member / val / non-member, masks, ref."""
    ds_dir = os.path.join(args.repo, 'dataset')
    sys.path.insert(0, ds_dir)
    from make_split import CLASSES, load_single_label_records   # same label filter as everyone

    split_path = os.path.join(ds_dir, 'splits', f'ptbxl_{LEVEL}_s{SEED}.json')
    meta = json.load(open(split_path.replace('.json', '.meta.json')))
    mask_path = os.path.join(ds_dir, 'splits', meta['shadow']['file'])
    for p, want in [(split_path, meta['sha256']), (mask_path, meta['shadow']['sha256'])]:
        got = sha256(p)
        log('sha256', os.path.basename(p), got[:12], 'OK' if got == want else 'MISMATCH')
        if got != want:
            sys.exit(f'{p} does not match .meta.json; do not use a private split')
    split = json.load(open(split_path))
    mk = np.load(mask_path)
    ids = mk['ids']

    ptbxl_dir = args.ptbxl_dir or find_ptbxl(args.data_root)
    df = load_single_label_records(ptbxl_dir)
    assert (df.index.values == ids).all(), 'single-label pool differs from the shared mask ids'
    y = df.label.values.astype(np.int64)

    cache = os.path.join(args.work, f'ptbxl_pool_{LOOKBACK + HORIZON}.npy')
    if os.path.exists(cache):
        x = np.load(cache, mmap_mode=None)
        log('loaded cached signals', x.shape)
    else:
        files = [os.path.join(ptbxl_dir, f) for f in df.filename_lr.values]
        log(f'reading {len(files)} records from {ptbxl_dir} ...')
        x = np.empty((len(files), N_LEADS, LOOKBACK + HORIZON), dtype=np.float32)
        t_old, t_new = np.linspace(0, 1, RAW_LEN), np.linspace(0, 1, LOOKBACK + HORIZON)
        with ThreadPoolExecutor(16) as ex:
            for s in range(0, len(files), 1000):                         # chunks keep RAM low
                raw = np.stack(list(ex.map(read_record, files[s:s + 1000]))).astype(np.float64)   # [n, 12, 1000]
                assert raw.shape[1:] == (N_LEADS, RAW_LEN), raw.shape
                c = interpolate.interp1d(t_old, raw, axis=-1)(t_new)
                x[s:s + len(c)] = (c - c.mean(-1, keepdims=True)) / (c.std(-1, keepdims=True) + 1e-8)   # z-score per lead per record
        os.makedirs(args.work, exist_ok=True)
        np.save(cache, x)
        log('signals cached', x.shape)
    assert np.isfinite(x).all()

    pos = {int(e): i for i, e in enumerate(ids)}
    idx = {k: np.array([pos[e] for e in split[k]]) for k in ('train', 'val', 'test')}
    K = int(meta['shadow']['K'])
    return dict(ids=ids, y=y, x=x, idx_m=idx['train'], idx_v=idx['val'], idx_n=idx['test'],
                masks=mk['masks'][:K], ref=np.flatnonzero(np.isin(ids, mk['ref_ids'])), K=K,
                meta=meta, classes=CLASSES)


# ---------------------------------------------------------------- model
def build_model(task):
    from transformers import PatchTSTConfig, PatchTSTForClassification, PatchTSTForPrediction
    if task == 'cls':
        return PatchTSTForClassification(PatchTSTConfig(**ARCH, num_targets=5))
    return PatchTSTForPrediction(PatchTSTConfig(**ARCH, prediction_length=HORIZON))


class Runner:
    """Holds the pool on the device and runs training / inference with the shared recipe."""

    def __init__(self, pool, task, device):
        self.task, self.dev = task, device
        self.x = torch.as_tensor(pool['x']).to(device)                 # [N, 12, 608]
        self.y = torch.as_tensor(pool['y']).to(device)
        self.bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8

    def amp(self):
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=self.bf16) if self.dev == 'cuda' \
            else torch.autocast(device_type='cpu', enabled=False)

    def forward(self, model, xb):
        inp = xb[..., :LOOKBACK].transpose(1, 2).contiguous()           # PatchTST wants [B, L, C]
        with self.amp():
            if self.task == 'cls':
                return model(past_values=inp).prediction_logits.float()
            return model(past_values=inp).prediction_outputs.float().transpose(1, 2)   # [B, 12, H]

    def loss(self, out, rows):
        if self.task == 'cls':
            return F.cross_entropy(out, self.y[rows])
        return F.mse_loss(out, self.x[rows][..., LOOKBACK:])

    @torch.no_grad()
    def predict(self, model, rows=None, bs=256):
        """cls -> logits [n, 5]; fcst -> predictions [n, 12, H]. Rows default to the whole pool."""
        model.eval()
        rows = torch.arange(len(self.x), device=self.dev) if rows is None else torch.as_tensor(rows, device=self.dev)
        out = [self.forward(model, self.x[rows[i:i + bs]]).cpu() for i in range(0, len(rows), bs)]
        return torch.cat(out)

    def fcst_mse(self, preds, rows=None):
        tgt = self.x[..., LOOKBACK:].cpu() if rows is None else self.x[torch.as_tensor(rows, device=self.dev)][..., LOOKBACK:].cpu()
        return ((preds.double() - tgt.double()) ** 2).mean(dim=(1, 2)).numpy()

    def train(self, rows, epochs, max_lr, seed, ckpt_path, key, eval_at=(), on_eval=None):
        """Train a fresh model on pool rows. Resumes from ckpt_path if its key matches."""
        control_randomness(seed)
        model = build_model(self.task).to(self.dev)
        rows = torch.as_tensor(np.asarray(rows), device=self.dev)
        steps = len(rows) // BATCH                                      # drop_last=True like the MOMENT runs
        opt = torch.optim.AdamW(model.parameters(), lr=max_lr / 25)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=epochs * steps)
        gen = torch.Generator().manual_seed(seed)
        start, state = 1, {}
        if ckpt_path and os.path.exists(ckpt_path):
            ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            if ck['key'] == key:
                model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
                gen.set_state(ck['gen']); start, state = ck['epoch'] + 1, ck.get('state', {})
                log(f'  resumed after epoch {ck["epoch"]}')
            del ck
        for ep in range(start, epochs + 1):
            model.train()
            t0, losses = time.time(), []
            perm = rows[torch.randperm(len(rows), generator=gen).to(self.dev)]
            for s in range(steps):
                b = perm[s * BATCH:(s + 1) * BATCH]
                opt.zero_grad(set_to_none=True)
                loss = self.loss(self.forward(model, self.x[b]), b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step(); sched.step()
                losses.append(loss.item())
            log(f'  ep {ep}/{epochs} train_loss {np.mean(losses):.4f} ({time.time() - t0:.0f}s)')
            if ep in eval_at and on_eval:
                on_eval(model, ep, float(np.mean(losses)), state)
            if ckpt_path and ep < epochs:
                save_atomic({'key': key, 'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                             'sched': sched.state_dict(), 'gen': gen.get_state(), 'state': state}, ckpt_path)
        if ckpt_path and os.path.exists(ckpt_path):
            os.remove(ckpt_path)
        return model, state


# ---------------------------------------------------------------- metrics + output
def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0


def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))


def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))],
                                sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]


def eval_epochs(epochs):
    return sorted({max(1, math.ceil(f * epochs)) for f in EVAL_FRACS})


def utility(runner, out, pool):
    """Utility of a target at one epoch from its pool outputs (cls logits or fcst per-record MSE)."""
    if runner.task == 'cls':
        hit = out.argmax(1).numpy() == pool['y']
        a_m, a_n, a_v = (float(hit[pool[k]].mean()) for k in ('idx_m', 'idx_n', 'idx_v'))
        return {'train_acc': a_m, 'test_acc': a_n, 'val_acc': a_v, 'gap': a_m - a_n}
    m_m, m_n, m_v = (float(out[pool[k]].mean()) for k in ('idx_m', 'idx_n', 'idx_v'))
    return {'train_mse': m_m, 'test_mse': m_n, 'val_mse': m_v, 'gap': m_n - m_m}


def write_result(args, pool, attack, sm, sn, util, extra, n_shadow=None):
    t01, t1, au = mia_metrics(sm, sn)
    c01, c1 = bootstrap_ci(sm, sn)
    out = {
        'model': 'patchtst', 'task': args.task, 'attack': attack, 'owner_tv': 2, 'level': LEVEL,
        'dataset': 'ptbxl', 'seed': SEED, 'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
        'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
        'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
        'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
        'utility': {k: v for k, v in util.items() if not k.startswith('val_')},
        'n_shadow': n_shadow,
        'extra': extra,
    }
    os.makedirs(args.results, exist_ok=True)
    fn = os.path.join(args.results, f'{attack}.json')
    with open(fn, 'w') as f:
        json.dump(out, f)
    log(f'saved {fn} | TPR@0.1% {t01:.4f} {np.round(c01, 4).tolist()} | TPR@1% {t1:.4f} | AUC {au:.4f}')
    return out


def common_extra(args, pool, recipe, tuning):
    e = {'epochs': recipe['epochs'], 'eval_epoch': recipe['epochs'], 'eval_epochs': eval_epochs(recipe['epochs']),
         'batch': BATCH, 'max_lr': recipe['max_lr'], 'backbone': BACKBONE, 'arch': ARCH,
         'optimizer': 'AdamW, OneCycleLR (lr0 = max_lr/25), grad clip 1.0, no early stopping',
         'tuning': tuning, 'split_sha256': pool['meta']['sha256'],
         'preprocess': f'interp 1000->{LOOKBACK + HORIZON}, z-score per lead per record, input = first {LOOKBACK}',
         'record_order': 'sorted ecg_id', 'deviations': DEVIATIONS, 'selection_note': SELECTION_NOTE}
    if args.task == 'fcst':
        e.update(lookback=LOOKBACK, horizon=HORIZON)
    return e


# ---------------------------------------------------------------- steps
def task_dir(args):
    d = os.path.join(args.work, f'patchtst_{args.task}')
    os.makedirs(d, exist_ok=True)
    return d


def recipe_key(args, pool, epochs, max_lr, extra=''):
    return f"patchtst|{args.task}|{json.dumps(ARCH, sort_keys=True)}|{LEVEL}|{SEED}|{epochs}|{BATCH}|{max_lr}|{pool['meta']['sha256'][:12]}{extra}"


def step_tune(args, pool, runner):
    path = os.path.join(task_dir(args), 'tuning.json')
    done = json.load(open(path)) if os.path.exists(path) else {'configs': []}
    grid = args.grid or DEFAULT_GRID
    assert len(grid) <= 6, 'dataset/README.md allows at most 6 configs per model and task'
    for epochs, lr in grid:
        if any(c['epochs'] == epochs and c['max_lr'] == lr for c in done['configs']):
            continue
        log(f'[tune {args.task}] epochs={epochs} max_lr={lr}')
        key = recipe_key(args, pool, epochs, lr, '|tune')
        model, _ = runner.train(pool['idx_m'], epochs, lr, SEED, os.path.join(task_dir(args), 'tune.ckpt.pt'), key)
        rows = np.r_[pool['idx_m'], pool['idx_v']]
        out = runner.predict(model, rows)
        nm = len(pool['idx_m'])
        if args.task == 'cls':
            hit = out.argmax(1).numpy() == pool['y'][rows]
            res = {'train_acc': float(hit[:nm].mean()), 'val_acc': float(hit[nm:].mean())}
        else:
            mse = runner.fcst_mse(out, rows)
            res = {'train_mse': float(mse[:nm].mean()), 'val_mse': float(mse[nm:].mean())}
        done['configs'].append({'epochs': epochs, 'max_lr': lr, **res})
        log(f'[tune {args.task}]', done['configs'][-1])
        json.dump(done, open(path, 'w'), indent=1)
    metric = 'val_acc' if args.task == 'cls' else 'val_mse'
    best = (max if args.task == 'cls' else min)(done['configs'], key=lambda c: c[metric])
    done['chosen'] = {'epochs': best['epochs'], 'max_lr': best['max_lr'], 'by': f'best {metric} (val only, no MIA scores)'}
    json.dump(done, open(path, 'w'), indent=1)
    log(f'[tune {args.task}] chosen', done['chosen'])
    return done


def get_recipe(args):
    path = os.path.join(task_dir(args), 'tuning.json')
    tuning = json.load(open(path)) if os.path.exists(path) else None
    if args.epochs and args.max_lr:
        recipe = {'epochs': args.epochs, 'max_lr': args.max_lr}
    elif tuning and 'chosen' in tuning:
        recipe = {k: tuning['chosen'][k] for k in ('epochs', 'max_lr')}
    else:
        sys.exit('no recipe: run the "tune" step first or pass --epochs and --max-lr')
    return recipe, (tuning['configs'] if tuning else 'manual recipe, no search')


def target_out_path(args, ep):
    return os.path.join(task_dir(args), f'target_ep{ep}.pt')


def step_target(args, pool, runner):
    recipe, tuning = get_recipe(args)
    E, lr = recipe['epochs'], recipe['max_lr']
    ev = eval_epochs(E)
    key = recipe_key(args, pool, E, lr)
    done_path = os.path.join(task_dir(args), 'target.done.json')
    if os.path.exists(done_path) and json.load(open(done_path))['key'] == key:
        log(f'[target {args.task}] already trained, skipping')
    else:
        log(f'[target {args.task}] epochs={E} max_lr={lr} eval at {ev}')

        def on_eval(model, ep, train_loss, state):
            out = runner.predict(model)                                  # whole pool, incl. val
            rec = {'key': key, 'epoch': ep}
            if args.task == 'cls':
                rec['logits'] = out
                u = utility(runner, out, pool)
            else:
                rec['preds'] = out.half()                                 # [N, 12, H], half precision to save disk
                rec['mse'] = runner.fcst_mse(out)
                u = utility(runner, rec['mse'], pool)
            save_atomic(rec, target_out_path(args, ep))
            state.setdefault('history', []).append({'epoch': ep, 'train_loss': train_loss, **u})
            log(f'  eval ep {ep}:', {k: round(v, 4) for k, v in u.items()})

        _, state = runner.train(pool['idx_m'], E, lr, SEED, os.path.join(task_dir(args), 'target.ckpt.pt'), key,
                                eval_at=ev, on_eval=on_eval)
        json.dump({'key': key, 'recipe': recipe, 'eval_epochs': ev, 'history': state.get('history', [])},
                  open(done_path, 'w'), indent=1)

    # baseline attacks at the last epoch
    rec = torch.load(target_out_path(args, E), weights_only=False)
    im, inn = pool['idx_m'], pool['idx_n']
    extra = common_extra(args, pool, recipe, tuning)
    if args.task == 'cls':
        logp = F.log_softmax(rec['logits'].double(), -1)
        p = logp.exp()
        ce = -logp.gather(1, torch.as_tensor(pool['y'])[:, None]).squeeze(1)
        scores = {'loss': (-ce).numpy(), 'conf': p.max(1).values.numpy(), 'entropy': (p * logp).sum(1).numpy()}
        util = utility(runner, rec['logits'], pool)
    else:
        scores = {'loss': -rec['mse']}
        util = utility(runner, rec['mse'], pool)
    for a, s in scores.items():
        r = write_result(args, pool, a, s[im], s[inn], util, extra)
        if a == 'loss' and r['auc'] <= 0.5:
            log('WARNING: Loss attack AUC <= 0.5; check the score direction and whether the model learned anything')


def step_shadows(args, pool, runner):
    recipe, _ = get_recipe(args)
    E, lr = recipe['epochs'], recipe['max_lr']
    for j in range(pool['K']):
        path = os.path.join(task_dir(args), f'shadow_k{j}.pt')
        key = recipe_key(args, pool, E, lr, f"|shadow{j}|{pool['meta']['shadow']['sha256'][:12]}")
        if os.path.exists(path) and torch.load(path, weights_only=False)['key'] == key:
            log(f'[shadow {j}] already trained, skipping'); continue
        t0 = time.time()
        log(f'[shadow {j}/{pool["K"]}] training on {pool["masks"][j].sum()} records, epochs={E} max_lr={lr}')
        model, _ = runner.train(np.flatnonzero(pool['masks'][j]), E, lr, SEED + 1 + j,
                                os.path.join(task_dir(args), f'shadow_k{j}.ckpt.pt'), key)
        out = runner.predict(model)
        rec = {'key': key}
        m = pool['masks'][j]
        if args.task == 'cls':
            rec['logits'] = out
            hit = out.argmax(1).numpy() == pool['y']
            log(f'[shadow {j}] acc in {hit[m].mean():.4f} out {hit[~m].mean():.4f}')
        else:
            rec['mse'] = runner.fcst_mse(out)
            log(f'[shadow {j}] mse in {rec["mse"][m].mean():.4f} out {rec["mse"][~m].mean():.4f}')
        save_atomic(rec, path)
        log(f'[shadow {j}] done in {(time.time() - t0) / 60:.1f} min')


def phi_p(logits, y):
    """phi = log(p/(1-p)) of the true class (stable), p = softmax prob of the true class."""
    z = logits.double(); y = torch.as_tensor(y).long()[:, None]
    phi = z.gather(1, y).squeeze(1) - torch.logsumexp(z.scatter(1, y, float('-inf')), 1)
    return phi.numpy(), torch.sigmoid(phi).numpy()


def group_stats(W, X):
    c = W.sum(0)
    mu = (W * X).sum(0) / c
    return mu, (W * (X - mu) ** 2).sum(0) / (c - 1)


def lira_scores(PHI_S, PHI_T, Min, idx_t, online):
    mu_in, v_in = group_stats(Min, PHI_S)
    mu_out, v_out = group_stats(1 - Min, PHI_S)
    s_in, s_out = np.sqrt(v_in[idx_t].mean()), np.sqrt(v_out[idx_t].mean())     # global std (few shadows per sample)
    x = PHI_T[idx_t]
    if online:
        return norm.logpdf(x, mu_in[idx_t], s_in) - norm.logpdf(x, mu_out[idx_t], s_out)
    return (x - mu_out[idx_t]) / s_out


def rmia_scores(P_S, P_T, Min, idx_t, ref, a, gamma, online):
    Mout = 1 - Min
    p_in = (Min * P_S).sum(0) / Min.sum(0)
    p_out = (Mout * P_S).sum(0) / Mout.sum(0)
    p_bar = 0.5 * (p_in + p_out) if online else 0.5 * ((1 + a) * p_out + (1 - a))
    ratio = P_T / p_bar
    z = ratio[np.intersect1d(ref, idx_t)]                                       # Z: ref ids the target was scored on
    return np.searchsorted(np.sort(z), ratio[idx_t] / gamma, side='right') / len(z)


def cross_fit(make_clf, X, y):
    score = np.zeros(len(y))
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y):
        score[te] = make_clf().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return score


def hgb():
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED)


def slope_last(v):
    t = np.arange(v.shape[-1], dtype=np.float64); t -= t.mean()
    return (v * t).sum(-1) / (t ** 2).sum()


def temporal_features_fcst(res):
    """Koren et al. style features of the forecast residual [n, 12, H]."""
    n = res.shape[0]
    mag = np.abs(np.fft.rfft(res, axis=-1))[..., 1:]                            # drop DC (= bias)
    bands = np.array_split(np.arange(mag.shape[-1]), N_BANDS)
    dft = np.stack([np.log1p(mag[..., b].mean(-1)) for b in bands], -1).reshape(n, -1)
    trend, drift = slope_last(res), slope_last(np.abs(res))
    lmse = np.log((res ** 2).mean(-1) + 1e-8)
    return np.concatenate([dft, trend, drift, lmse], 1)


def per_epoch_cls(logits, labels):
    logp = F.log_softmax(logits.double(), -1)
    lab = torch.as_tensor(labels)[:, None]
    ce = -logp.gather(1, lab).squeeze(1)
    ent = -(logp.exp() * logp).sum(1)
    true = logits.double().gather(1, lab).squeeze(1)
    other = logits.double().scatter(1, lab, float('-inf')).max(1).values
    return ce.numpy(), ent.numpy(), (true - other).numpy()


def step_attacks(args, pool, runner):
    recipe, tuning = get_recipe(args)
    E = recipe['epochs']
    ev = eval_epochs(E)
    im, iv, inn = pool['idx_m'], pool['idx_v'], pool['idx_n']
    idx_t = np.r_[im, inn]
    nm = len(im)
    extra = common_extra(args, pool, recipe, tuning)
    shadow_meta = pool['meta']['shadow']
    a_rmia, g_rmia = shadow_meta['rmia_defaults']['a'], shadow_meta['rmia_defaults']['gamma']
    tgt = torch.load(target_out_path(args, E), weights_only=False)
    K = pool['K']
    shadows = [torch.load(os.path.join(task_dir(args), f'shadow_k{j}.pt'), weights_only=False) for j in range(K)]
    Min = pool['masks'].astype(float)

    if args.task == 'cls':
        PHI_S, P_S = map(np.stack, zip(*[phi_p(s['logits'], pool['y']) for s in shadows]))
        PHI_T, P_T = phi_p(tgt['logits'], pool['y'])
        util = utility(runner, tgt['logits'], pool)
        rmia_note = ''
    else:
        S_mse = np.stack([s['mse'] for s in shadows]).astype(np.float64)
        T_mse = tgt['mse'].astype(np.float64)
        PHI_S, PHI_T = -np.log(S_mse + 1e-8), -np.log(T_mse + 1e-8)
        tau = float(np.median(S_mse[~pool['masks']]))                          # attacker-side only
        P_S, P_T = np.exp(-S_mse / tau), np.exp(-T_mse / tau)
        util = utility(runner, T_mse, pool)
        rmia_note = f' p=exp(-MSE/tau) tau={tau:.5f}'
        sens = {}
        for tm in (0.5, 1.0, 2.0):                                             # diagnostic only, files use x1.0
            for online in (False, True):
                s = rmia_scores(np.exp(-S_mse / (tau * tm)), np.exp(-T_mse / (tau * tm)), Min, idx_t, pool['ref'], a_rmia, g_rmia, online)
                sens[f"{'online' if online else 'offline'}_x{tm}"] = mia_metrics(s[:nm], s[nm:])
        extra = {**extra, 'tau': tau, 'rmia_tau_sensitivity': sens}

    scores = {'lira': lira_scores(PHI_S, PHI_T, Min, idx_t, True),
              'lira_offline': lira_scores(PHI_S, PHI_T, Min, idx_t, False),
              'rmia': rmia_scores(P_S, P_T, Min, idx_t, pool['ref'], a_rmia, g_rmia, False),
              'rmia_online': rmia_scores(P_S, P_T, Min, idx_t, pool['ref'], a_rmia, g_rmia, True)}
    method = {'lira': 'LiRA online, global variance', 'lira_offline': 'LiRA offline z-score, global variance',
              'rmia': f'RMIA offline a={a_rmia} gamma={g_rmia}{rmia_note}', 'rmia_online': f'RMIA online gamma={g_rmia}{rmia_note}'}
    for a, s in scores.items():
        write_result(args, pool, a, s[:nm], s[nm:], util,
                     {**extra, 'method': method[a], 'shadow_masks_sha256': shadow_meta['sha256']}, n_shadow=K)

    # Temporal (gray-box, cross-fitted HistGradientBoosting)
    y_t = np.r_[np.ones(nm), np.zeros(len(inn))]
    if args.task == 'cls':
        recs = {ep: torch.load(target_out_path(args, ep), weights_only=False)['logits'] for ep in ev}
        lab = pool['y'][idx_t]
        ce, ent, mar = (np.stack(z, 1) for z in zip(*[per_epoch_cls(recs[ep][idx_t], lab) for ep in ev]))
        t = np.asarray(ev, dtype=np.float64); t -= t.mean()
        slope_ep = lambda v: (v * t).sum(1) / (t ** 2).sum()
        X = np.concatenate([ce, ent, mar, np.diff(ce, axis=1), np.stack([slope_ep(ce), slope_ep(mar)], 1)], 1)
        Xf = np.stack([ce[:, -1], ent[:, -1], mar[:, -1]], 1)
        loss_all = -ce[:, -1]
        makers = {'temporal': (hgb, X),
                  'temporal_lr': (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X),
                  'final_epoch_clf': (hgb, Xf), 'loss_only_clf': (hgb, loss_all[:, None])}
        t_extra = {'variant': 'training-dynamics', 'eval_epochs': ev,
                   'features': 'per-epoch CE/entropy/margin + consecutive CE change + CE and margin slope over checkpoints',
                   'threat_model': 'gray-box: intermediate checkpoints, true labels and a labelled member/non-member subset'}
    else:
        tgt_y = pool['x'][idx_t][..., LOOKBACK:].astype(np.float32)
        res = tgt_y - tgt['preds'][idx_t].float().numpy()
        X = temporal_features_fcst(res)
        loss_all = -(res ** 2).mean((1, 2))
        makers = {'temporal': (hgb, X),
                  'temporal_lr': (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X),
                  'loss_only_clf': (hgb, loss_all[:, None])}
        t_extra = {'variant': 'residual DFT (Koren et al.)', 'n_bands': N_BANDS,
                   'features': f'DFT {N_BANDS} bands + trend + drift + per-lead log MSE of the residual',
                   'threat_model': 'gray-box: a labelled member/non-member subset'}
    S, M = {}, {}
    for name, (mk, feats) in makers.items():
        S[name] = cross_fit(mk, feats, y_t)
        M[name] = mia_metrics(S[name][:nm], S[name][nm:])
    M['loss (raw)'] = mia_metrics(loss_all[:nm], loss_all[nm:])
    log('[temporal] ablation\n' + pd.DataFrame(M, index=['TPR@0.1%', 'TPR@1%', 'AUC']).T.round(4).to_string())
    write_result(args, pool, 'temporal', S['temporal'][:nm], S['temporal'][nm:], util,
                 {**extra, **t_extra, 'classifier': 'HistGradientBoosting(max_iter=200, lr=0.05)', 'n_folds': N_FOLDS,
                  'ablation': {k: {'tpr_at_0.1fpr': v[0], 'tpr_at_1fpr': v[1], 'auc': v[2]} for k, v in M.items()}})
    plot_roc(args)


def plot_roc(args):
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, ax = plt.subplots(figsize=(6, 6))
    rows = []
    for a in ['loss', 'conf', 'entropy', 'lira', 'lira_offline', 'rmia', 'rmia_online', 'temporal']:
        p = os.path.join(args.results, f'{a}.json')
        if not os.path.exists(p):
            continue
        r = json.load(open(p))
        if r.get('scores_member') is None:
            continue
        sm, sn = np.array(r['scores_member']), np.array(r['scores_nonmember'])
        fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
        ax.plot(fpr, tpr, label=f"{a} (TPR@0.1% {r['tpr_at_0.1fpr']:.3f}, AUC {r['auc']:.3f})")
        rows.append({'attack': a, 'tpr@0.1%': r['tpr_at_0.1fpr'], 'tpr@1%': r['tpr_at_1fpr'], 'auc': r['auc']})
    ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=7)
    ax.set_title(f'PatchTST PTB-XL {args.task}, {LEVEL}-level')
    out = os.path.join(args.work, f'patchtst_{args.task}_roc.png')
    fig.savefig(out, dpi=150, bbox_inches='tight')
    log('ROC saved to', out)
    if rows:
        log('\n' + pd.DataFrame(rows).round(4).to_string(index=False))


def step_smoke(args, pool, runner):
    """1 epoch on 320 members: checks data, model, loss direction and the JSON writer without the full cost."""
    rows = pool['idx_m'][:320]
    model, _ = runner.train(rows, 1, 1e-3, SEED, None, 'smoke')
    out = runner.predict(model, np.r_[rows, pool['idx_n'][:320]])
    log('smoke ok, output shape', tuple(out.shape), 'device', runner.dev)


# ---------------------------------------------------------------- cli
def main():
    here = os.path.dirname(os.path.abspath(__file__))
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('step', choices=['tune', 'target', 'shadows', 'attacks', 'all', 'smoke', 'plot'])
    ap.add_argument('--task', choices=['cls', 'fcst'], required=True)
    ap.add_argument('--repo', default=os.path.abspath(os.path.join(here, '..', '..')), help='MIA repo root')
    ap.add_argument('--data-root', default='/kaggle/input', help='searched for ptbxl_database.csv')
    ap.add_argument('--ptbxl-dir', default=None, help='folder that contains ptbxl_database.csv')
    ap.add_argument('--work', default=None, help='checkpoints, cached signals, model outputs (default: <repo>/work)')
    ap.add_argument('--results', default=None, help='default: <repo>/results/patchtst/<task>')
    ap.add_argument('--epochs', type=int, default=None, help='override the tuned recipe')
    ap.add_argument('--max-lr', type=float, default=None, help='override the tuned recipe')
    ap.add_argument('--grid', default=None, help='tune grid "epochs:lr,..." (max 6), default ' +
                    ','.join(f'{e}:{l:g}' for e, l in DEFAULT_GRID))
    ap.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    args = ap.parse_args()
    args.work = args.work or os.path.join(args.repo, 'work')
    args.results = args.results or os.path.join(args.repo, 'results', 'patchtst', args.task)
    if args.grid:
        args.grid = [(int(e), float(l)) for e, l in (g.split(':') for g in args.grid.split(','))]
    os.makedirs(args.work, exist_ok=True)

    if args.step == 'plot':
        return plot_roc(args)
    if args.device == 'cpu' and args.step != 'smoke':
        log('WARNING: no GPU found, training will be very slow')
    elif args.device == 'cuda':
        log('GPU', torch.cuda.get_device_name(0))
    pool = load_pool(args)
    log(f"pool {len(pool['ids'])} | member {len(pool['idx_m'])} val {len(pool['idx_v'])} non-member {len(pool['idx_n'])} | K {pool['K']}")
    runner = Runner(pool, args.task, args.device)
    steps = {'tune': step_tune, 'target': step_target, 'shadows': step_shadows, 'attacks': step_attacks, 'smoke': step_smoke}
    for s in (['tune', 'target', 'shadows', 'attacks'] if args.step == 'all' else [args.step]):
        t0 = time.time()
        steps[s](args, pool, runner)
        log(f'== {s} ({args.task}) finished in {(time.time() - t0) / 60:.1f} min')


if __name__ == '__main__':
    main()

In [ ]:
# transformers (có PatchTST) có sẵn trên Kaggle; cài thêm nếu thiếu
try:
    from transformers import PatchTSTForClassification
except ImportError:
    !pip install -q -U transformers
!pip install -q wfdb

## 2. Kiểm tra nhanh (đọc dữ liệu, tạo cache tín hiệu, 1 epoch nhỏ)

In [ ]:
SCRIPT = f'{REPO}/experiment/patchtst/patchtst_mia.py'
WORK = '/kaggle/working/work'
!python {SCRIPT} smoke --task cls --work {WORK} 2>&1 | grep -v Warning

## 3. Chạy đủ pipeline cho cls và fcst

Log ghi vào `/kaggle/working/work/log_<task>.txt`. Nếu session bị ngắt, chạy lại cell này: script tự bỏ qua bước đã xong và resume từ checkpoint.
Muốn đổi lưới tune thì thêm `--grid "20:1e-4,50:5e-4"` (tối đa 6 cấu hình).

In [ ]:
import subprocess, time, torch
n_gpu = torch.cuda.device_count()
print('GPUs:', n_gpu)

def launch(task, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONWARNINGS='ignore')
    log = open(f'{WORK}/log_{task}.txt', 'a')
    return subprocess.Popen(['python', SCRIPT, 'all', '--task', task, '--work', WORK],
                            stdout=log, stderr=subprocess.STDOUT, env=env)

def wait(procs):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(120)
        for t in procs:
            tail = open(f'{WORK}/log_{t}.txt').read().strip().splitlines()[-1:]
            print(t, '|', tail[0] if tail else '')
    for t, p in procs.items():
        print(t, 'exit code', p.returncode)

if n_gpu >= 2:
    wait({'cls': launch('cls', 0), 'fcst': launch('fcst', 1)})
else:
    for t in ('cls', 'fcst'):
        wait({t: launch(t, 0)})

In [ ]:
# log cuối của từng task (bảng ablation Temporal, TPR/AUC của từng attack)
for t in ('cls', 'fcst'):
    print(f'===== {t}'); print(''.join(open(f'{WORK}/log_{t}.txt').readlines()[-40:]))

## 4. Tổng hợp kết quả

In [ ]:
import json, pandas as pd
rows = []
for f in sorted(glob.glob(f'{REPO}/results/patchtst/*/*.json')):
    r = json.load(open(f))
    if r.get('tpr_at_0.1fpr') is None:
        continue
    rows.append({'task': r['task'], 'attack': r['attack'], 'TPR@0.1%': r['tpr_at_0.1fpr'],
                 'CI@0.1%': r['bootstrap_ci']['tpr_at_0.1fpr'], 'TPR@1%': r['tpr_at_1fpr'],
                 'CI@1%': r['bootstrap_ci']['tpr_at_1fpr'], 'AUC': r['auc'], **r['utility']})
pd.DataFrame(rows).round(4)

In [ ]:
from IPython.display import Image, display
for t in ('cls', 'fcst'):
    p = f'{WORK}/patchtst_{t}_roc.png'
    if os.path.exists(p): display(Image(p))

In [ ]:
# Gói kết quả theo đúng cấu trúc repo: giải nén zip vào GỐC repo MIA (chọn "Extract here", không tạo thư mục mới)
#   results/patchtst/<task>/*.json                     -> kết quả MIA
#   experiment/patchtst/run_artifacts/<task>/...       -> tuning, utility theo epoch, log, ROC
import shutil
STAGE = '/kaggle/working/_zip'
shutil.rmtree(STAGE, ignore_errors=True)
shutil.copytree(f'{REPO}/results/patchtst', f'{STAGE}/results/patchtst')
for t in ('cls', 'fcst'):
    d = f'{STAGE}/experiment/patchtst/run_artifacts/{t}'
    os.makedirs(d, exist_ok=True)
    for src, dst in [(f'{WORK}/patchtst_{t}/tuning.json', 'tuning.json'), (f'{WORK}/patchtst_{t}/target.done.json', 'target.done.json'),
                     (f'{WORK}/patchtst_{t}_roc.png', 'roc.png'), (f'{WORK}/log_{t}.txt', 'log.txt')]:
        if os.path.exists(src): shutil.copy(src, f'{d}/{dst}')
if os.path.exists('/kaggle/working/patchtst_results.zip'): os.remove('/kaggle/working/patchtst_results.zip')
!cd {STAGE} && zip -qr /kaggle/working/patchtst_results.zip results experiment
!unzip -l /kaggle/working/patchtst_results.zip